# 13 Class 2 vs 3: stress tests

Three ways to try to break the result from 12: stricter twin grouping, a negative control, and a 40 Hz low-pass.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, RANDOM_STATE
from src.cv import load_fold_table
from src.evaluate import cross_validate_segments
from src.features import FEATURES, SIZE_FEATURES, SHAPE_FEATURES

feat = pd.read_csv(PROCESSED / "features_dev.csv")
table = load_fold_table("cv_folds_allclass.csv")
data = feat[feat["y"].isin([2, 3])].reset_index(drop=True)          # Kaggle classes 2 and 3, development set only
assert set(data["segment"]) <= set(table["segment"])

models = {
    "chance": lambda: DummyClassifier(strategy="prior"),
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random forest": lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=-1),
}

## Stricter grouping

Re-make the groups with lower similarity thresholds (strong / weak), build new folds from each, and re-score. Groups come from all 500 segments' similarity, then are applied to the 160.

In [2]:
from src.cv import make_group_fold_table
from src.data import load_raw, feature_cols
from src.groups import similarity_matrix, make_groups

raw = load_raw()
names_all, M_all = similarity_matrix(raw, feature_cols(raw))          # similarity between all 500 segments

settings = {"current (0.7 / 0.4)": (0.7, 0.4), "stricter (0.6 / 0.3)": (0.6, 0.3), "strictest (0.5 / 0.2)": (0.5, 0.2)}
use = {"all 23 features": FEATURES, "shape only (15)": SHAPE_FEATURES}
diag, auc = [], {}
for sname, (strong, weak) in settings.items():
    group_of = make_groups(names_all, M_all, strong=strong, weak=weak)
    d = data.assign(group=data["segment"].map(group_of).to_numpy())
    new_table = make_group_fold_table(d, group_of)
    size = d.groupby("group").size()
    f0 = d["segment"].map(new_table.set_index("segment")["repeat_0"])
    c = pd.crosstab(f0, d["y"])
    diag.append({"setting": sname, "groups": d["group"].nunique(), "biggest group": int(size.max()),
                 "groups with both classes": int((d.groupby("group")["y"].nunique() > 1).sum()),
                 "segments per fold": f"{c.sum(axis=1).min()}-{c.sum(axis=1).max()}"})
    for fname, cols in use.items():
        for mname in ["logistic regression", "random forest"]:
            auc[(fname, mname, sname)] = cross_validate_segments(models[mname], d, cols, new_table, positive=2)["auc_fold"].mean()

display(pd.DataFrame(diag))
display(pd.Series(auc).unstack().round(3))

,setting,groups,biggest group,groups with both classes,segments per fold
0,current (0.7 / 0.4),76,9,1,15-18
1,stricter (0.6 / 0.3),64,9,1,15-17
2,strictest (0.5 / 0.2),58,9,5,15-17


current (0.7 / 0.4)  \
all 23 features logistic regression                0.740   
                random forest                      0.762   
shape only (15) logistic regression                0.761   
                random forest                      0.768   

                                     stricter (0.6 / 0.3)  \
all 23 features logistic regression                 0.703   
                random forest                       0.739   
shape only (15) logistic regression                 0.737   
                random forest                       0.740   

                                     strictest (0.5 / 0.2)  
all 23 features logistic regression                  0.708  
                random forest                        0.739  
shape only (15) logistic regression                  0.742  
                random forest                        0.744

## Negative control

Features that should carry nothing about the class: power in the 45 to 55 Hz mains line and in the 60 to 80 Hz noise floor. If these score well above chance, something other than brain signal is leaking in.

In [3]:
from scipy import signal as sps
from src.config import SAMPLING_RATE_HZ
from src.features import reconstruct_segments

raw_23 = raw[raw["segment"].isin(data["segment"])]
names_c, sigs = reconstruct_segments(raw_23, feature_cols(raw))

def control_features(x):
    xc = x - x.mean()
    f, p = sps.welch(xc, fs=SAMPLING_RATE_HZ, nperseg=512)
    df_hz = f[1] - f[0]
    band = lambda lo, hi: p[(f >= lo) & (f < hi)].sum() * df_hz
    total = p.sum() * df_hz
    return {"logpow_mains_45_55": np.log10(band(45, 55) + 1e-12), "logpow_noise_60_80": np.log10(band(60, 80) + 1e-12),
            "rel_mains": band(45, 55) / total, "rel_noise": band(60, 80) / total}

ctrl = pd.DataFrame([control_features(x) for x in sigs])
ctrl.insert(0, "segment", names_c)
data_c = data[["segment", "y", "group"]].merge(ctrl, on="segment")
ctrl_cols = ["logpow_mains_45_55", "logpow_noise_60_80", "rel_mains", "rel_noise"]

for mname in ["logistic regression", "random forest"]:
    r = cross_validate_segments(models[mname], data_c, ctrl_cols, table, positive=2)
    print(f"{mname:20s} negative-control AUC {r['auc_fold'].mean():.3f} (sd {r['auc_fold'].std():.3f})")

logistic regression  negative-control AUC 0.386 (sd 0.030)


random forest        negative-control AUC 0.528 (sd 0.040)


## 40 Hz low-pass

Smooth out everything above 40 Hz (zero-phase Butterworth), recompute all features, and re-score. Tests whether the result depends on high-frequency noise.

In [4]:
from scipy.signal import butter, sosfiltfilt
from src.features import segment_features

sos = butter(4, 40, btype="low", fs=SAMPLING_RATE_HZ, output="sos")
filtered = np.stack([sosfiltfilt(sos, x) for x in sigs])                      # the 160 reconstructed segments
feats_f = pd.DataFrame([segment_features(x) for x in filtered])[FEATURES]
feats_f.insert(0, "segment", names_c)
data_f = data[["segment", "y", "group"]].merge(feats_f, on="segment")

orig = data.set_index("segment").loc[data_f["segment"]]
for f in ["hjorth_complexity", "hjorth_mobility", "line_length", "zero_crossing_rate", "spectral_centroid"]:
    print(f"{f:20s} median before {orig[f].median():8.3f} | after low-pass {data_f[f].median():8.3f}")

rows = {}
for fname, cols in {"all 23 features": FEATURES, "shape only (15)": SHAPE_FEATURES}.items():
    for mname in ["logistic regression", "random forest"]:
        rows[(fname, mname)] = cross_validate_segments(models[mname], data_f, cols, table, positive=2)["auc_fold"].mean()
rows[("hjorth_complexity alone", "logistic regression")] = cross_validate_segments(
    models["logistic regression"], data_f, ["hjorth_complexity"], table, positive=2)["auc_fold"].mean()
display(pd.Series(rows, name="AUC after 40 Hz low-pass").round(3))

hjorth_complexity    median before    3.840 | after low-pass    2.743
hjorth_mobility      median before    0.204 | after low-pass    0.188
line_length          median before    7.127 | after low-pass    6.610
zero_crossing_rate   median before    0.064 | after low-pass    0.058
spectral_centroid    median before    3.848 | after low-pass    3.832


all 23 features          logistic regression    0.717
                         random forest          0.764
shape only (15)          logistic regression    0.762
                         random forest          0.794
hjorth_complexity alone  logistic regression    0.651
Name: AUC after 40 Hz low-pass, dtype: float64

## Findings

- **Stricter grouping:** 76 / 64 / 58 groups (current / stricter / strictest), with 1 / 1 / 5 holding both classes. AUC (current / stricter / strictest): logistic all 23 0.740 / 0.703 / 0.708, forest all 23 0.762 / 0.739 / 0.739, logistic shape 0.761 / 0.737 / 0.742, forest shape 0.768 / 0.740 / 0.744. Scores fall by 0.02 to 0.04, about the size of the noise yardstick, and stay above chance. (The "current" column uses re-made folds, so it differs a little from 12.)
- **Negative control:** mains and noise-floor features give 0.386 (logistic) and 0.528 (forest). Nothing there.
- **40 Hz low-pass:** logistic all 23 0.717, forest all 23 0.764, logistic shape 0.762, forest shape 0.794, complexity alone 0.651. Little changes.

## Caveats

- These tests rule out some explanations (weak twins, mains noise, high-frequency noise). They do not rule out patient or site overlap.
- Even the strictest grouping is a similarity-based guess. Scores are upper bounds.
- The stricter-grouping folds are new folds, so the noise yardstick (about 0.03) applies.